# Qwen3.8-27BをColabで動かす

Qwenの **Qwen3.8-27B** を4bit量子化で読み込み、通常チャットとGradio UIを実行します。

添付のPhi-4 Notebookからできるだけ構成を変えず、環境準備 → Google Drive cache → 4bitモデル読み込み → 日本語テスト → Gradio UIの流れを維持しています。

> 27B denseモデルのため、まずGoogle Colab Proの大容量GPUを想定します。


In [ ]:
# =========================================
# コード1 実行環境の準備
# =========================================
!nvidia-smi -L || echo "No GPU"
!python -V
%pip -q install -U transformers accelerate bitsandbytes

import torch, transformers
if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを有効にしてください。")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# =========================================
# コード2 Google Driveとキャッシュ設定
# =========================================
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
from pathlib import Path
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/LocalLLM")
USE_DRIVE_CACHE = True
DISABLE_XET = False
OFFLINE_MODE = False

CACHE_DIR = PROJECT_DIR/"Program"/"hf_cache_qwen38_27b" if USE_DRIVE_CACHE else Path("/content/hf_cache_qwen38_27b")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)
if DISABLE_XET: os.environ["HF_HUB_DISABLE_XET"]="1"
if OFFLINE_MODE: os.environ["HF_HUB_OFFLINE"]="1"

usage=shutil.disk_usage(CACHE_DIR)
print("CACHE_DIR:", CACHE_DIR)
print(f"free space: {usage.free/1024**3:.1f} GB")
if USE_DRIVE_CACHE and usage.free < 70*1024**3:
    print("WARNING: 70GB程度以上の空き容量を推奨します。")


- 時間がかかります。

In [ ]:
# =========================================
# コード3 Qwen3.8-27Bモデルと応答生成関数
# =========================================
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen3.8-27B"

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    cache_dir=str(CACHE_DIR),
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)

model.eval()

INPUT_DEVICE = model.get_input_embeddings().weight.device


@torch.inference_mode()
def chat_generate(
    messages,
    max_new_tokens=256,
    do_sample=False,
    temperature=0.7,
    top_p=0.8,
):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,

        # Qwen3系ではここが重要
        enable_thinking=False,

        return_dict=True,
        return_tensors="pt",
    ).to(INPUT_DEVICE)

    generation_kwargs = {
        **inputs,
        "max_new_tokens": int(max_new_tokens),
        "do_sample": bool(do_sample),
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "repetition_penalty": 1.05,
        "use_cache": True,
    }

    if do_sample:
        generation_kwargs["temperature"] = float(temperature)
        generation_kwargs["top_p"] = float(top_p)

    outputs = model.generate(
        **generation_kwargs
    )

    generated_ids = outputs[
        0,
        inputs["input_ids"].shape[-1]:
    ]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return answer


print("model loaded:", MODEL_ID)
print("input device:", INPUT_DEVICE)
print("4bit:", getattr(model, "is_loaded_in_4bit", False))
print("compute dtype:", COMPUTE_DTYPE)

In [ ]:
# =========================================
# コード4 動作確認
# =========================================
messages=[
 {"role":"system","content":"あなたは日本語で簡潔に答える親切なアシスタントです。"},
 {"role":"user","content":"日本語で1文だけ自己紹介してください。"},
]
print(chat_generate(messages,max_new_tokens=96,do_sample=False))


In [ ]:
# =========================================
# コード5 日本語推論テスト
# =========================================
messages=[
 {"role":"system","content":"あなたは日本語で分かりやすく答えるアシスタントです。"},
 {"role":"user","content":"定価3000円の本を20%引きで購入し、その価格に10%の消費税がかかります。支払額を計算過程とともに簡潔に説明してください。"},
]
print(chat_generate(messages,max_new_tokens=256,do_sample=False))


In [ ]:
# =========================================
# コード6 Gradioを用いたローカルLLMチャットUI
# =========================================
import gradio as gr
import inspect, torch

def make_messages_chatbot(**kwargs):
    if "type" in inspect.signature(gr.Chatbot).parameters:
        kwargs["type"]="messages"
    return gr.Chatbot(**kwargs)

print("gradio:",gr.__version__)

def gr_chat(history,user_msg):
    history=history or []
    user_msg=(user_msg or "").strip()
    if not user_msg:
        return history,"",history
    messages=history[-8:]+[{"role":"user","content":user_msg}]
    try:
        reply=chat_generate(messages,max_new_tokens=256,do_sample=False)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        reply="CUDA out of memory が発生しました。会話履歴をClearするか、ランタイムを再起動してください。"
    except Exception as e:
        reply=f"{type(e).__name__}: {e}"
    new_history=history+[{"role":"user","content":user_msg},{"role":"assistant","content":reply}]
    return new_history,"",new_history

with gr.Blocks(title="Qwen3.8-27B Local Chat") as chat_demo:
    gr.Markdown("## Qwen3.8-27B — Local 4bit Chat")
    chatbot=make_messages_chatbot(label="Chat",show_label=False,sanitize_html=True)
    chat_state=gr.State([])
    user_box=gr.Textbox(placeholder="質問を入力してください。",label="")
    with gr.Row():
        send_btn=gr.Button("Send",variant="primary")
        clear_btn=gr.Button("Clear")
    send_btn.click(gr_chat,inputs=[chat_state,user_box],outputs=[chat_state,user_box,chatbot],queue=False)
    clear_btn.click(lambda:([],"",[]),outputs=[chat_state,user_box,chatbot])

print("WARNING: share=Trueで公開URLが作成されます。")
chat_demo.launch(share=True,inline=True,debug=False)


In [ ]:
# =========================================
# コード7 GPUメモリ使用量を確認
# =========================================
import torch
print("GPU allocated: %.2f GB"%(torch.cuda.memory_allocated()/1024**3))
print("GPU reserved : %.2f GB"%(torch.cuda.memory_reserved()/1024**3))
free,total=torch.cuda.mem_get_info()
print("GPU free      : %.2f GB"%(free/1024**3))
print("GPU total     : %.2f GB"%(total/1024**3))


In [ ]:
# =========================================
# コード8 bitsandbytesの4bit量子化設定を確認
# =========================================
print(bnb_config)
